<a href="https://colab.research.google.com/github/t6niskoppel/Optimization-for-Robot-Motion-Planning-and-Control-assignment3/blob/main/colab/assignment_3_tonis_koppel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimization for Robot Motion Planning and Control
Assignment 3: MPC on IR_SIM using JAX-based Gradient Descent, Nesterov, and CEM.

In [ ]:
import os
import sys
import shutil

# 1. Full Reset
repo_path = '/content/opt_irsim'
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)

# 2. Clean Install
print('Installing ir-sim and dependencies...')
# !pip uninstall -y ir-sim
!pip install -q ir-sim[all] open3d

# 3. Clone fresh
!git clone -q https://github.com/t6niskoppel/Optimization-for-Robot-Motion-Planning-and-Control-assignment3.git {repo_path}

# 4. Handle double-nesting
nested = os.path.join(repo_path, 'opt_irsim')
if os.path.exists(nested):
    for item in os.listdir(nested):
        shutil.move(os.path.join(nested, item), repo_path)
    os.rmdir(nested)

# 5. Set working directory
%cd {repo_path}
if repo_path not in sys.path:
    sys.path.append(repo_path)

print('\n' + '='*50)
print('SETUP COMPLETE. RESTARTING RUNTIME TO REGISTER PACKAGES...')
print('='*50)

# 6. Force Restart
import os
os.kill(os.getpid(), 9)

Installing ir-sim and dependencies...
Found existing installation: ir-sim 2.9.3
Uninstalling ir-sim-2.9.3:
  Successfully uninstalled ir-sim-2.9.3


## 1. Physics Model and Cost Functions
Implementing the Unicycle model and trajectory rollouts using JAX.

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, grad, lax
from jax.random import PRNGKey, split, normal

@jit
def motion_model(state, action, dt):
    new_theta = state[2] + action[1] * dt
    new_x = state[0] + action[0] * jnp.cos(new_theta) * dt
    new_y = state[1] + action[0] * jnp.sin(new_theta) * dt
    return jnp.array([new_x, new_y, new_theta])

@jit
def compute_rollout(actions, initial_state, dt):
    def scan_fn(state, action):
        next_state = motion_model(state, action, dt)
        return next_state, next_state
    _, trajectory = lax.scan(scan_fn, initial_state, actions)
    return trajectory

## 2. MPC Planner Implementation
Implementing Gradient Descent, Nesterov, and CEM optimizers.

In [ ]:
class MPCPlanner:
    def __init__(self, dt=0.1, horizon=20, method='GD'):
        self.dt, self.horizon, self.method = dt, horizon, method
        self.key = PRNGKey(0)

    def cost_fn(self, actions, initial_state, goal_local, pcd):
        traj = compute_rollout(actions, initial_state, self.dt)
        dist_cost = jnp.linalg.norm(traj[-1, :2] - goal_local[:2])
        return dist_cost

    def solve(self, initial_state, goal_local, pcd):
        actions = jnp.zeros((self.horizon, 2))
        if self.method == 'GD':
            lr = 0.01
            for _ in range(10):
                actions -= lr * grad(self.cost_fn)(actions, initial_state, goal_local, pcd)
        elif self.method == 'Nesterov':
            v, mu, lr = jnp.zeros_like(actions), 0.9, 0.01
            for _ in range(10):
                g = grad(self.cost_fn)(actions + mu * v, initial_state, goal_local, pcd)
                v = mu * v - lr * g
                actions += v
        elif self.method == 'CEM':
            mean, std, n_samples, n_elite = actions, jnp.ones_like(actions)*0.5, 50, 5
            for _ in range(3):
                self.key, subkey = split(self.key)
                samples = mean + std * normal(subkey, (n_samples, self.horizon, 2))
                costs = vmap(lambda a: self.cost_fn(a, initial_state, goal_local, pcd))(samples)
                elite = samples[jnp.argsort(costs)[:n_elite]]
                mean, std = jnp.mean(elite, axis=0), jnp.std(elite, axis=0) + 1e-4
            actions = mean
        return actions[0], compute_rollout(actions, initial_state, self.dt)

## 3. Evaluation Framework
Iterate through BARN environments and record metrics.

In [ ]:
!python test.py

### Batch Evaluation Script
This script will iterate through the BARN environments and run the simulation for different planner configurations.

In [ ]:
import os
import pandas as pd
import time
from ir_sim.env import EnvBase

def run_evaluation(planner_type, max_envs=10):
    barn_dir = 'barn_envs/'
    env_files = sorted([f for f in os.listdir(barn_dir) if f.endswith('.yaml')])[:max_envs]
    results = []
    for env_file in env_files:
        env = EnvBase(os.path.join(barn_dir, env_file), plot=False, full=False)
        # Simulation loop logic here...
        env.close()
    print('Evaluation logic skeleton ready.')

### JAX-based MPC Implementation
We will implement the unicycle motion model and the cost functions using JAX to enable automatic differentiation for Gradient Descent.

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, grad, lax
from jax.random import PRNGKey, split, normal

@jit
def motion_model(state, action, dt):
    new_theta = state[2] + action[1] * dt
    new_x = state[0] + action[0] * jnp.cos(new_theta) * dt
    new_y = state[1] + action[0] * jnp.sin(new_theta) * dt
    return jnp.array([new_x, new_y, new_theta])

@jit
def compute_rollout(actions, initial_state, dt):
    def scan_fn(state, action):
        next_state = motion_model(state, action, dt)
        return next_state, next_state
    _, trajectory = lax.scan(scan_fn, initial_state, actions)
    return trajectory

### Planner Implementation
Implementing Gradient Descent, Nesterov, and CEM as requested.

In [ ]:
class MPCPlanner:
    def __init__(self, dt=0.1, horizon=20, method='GD'):
        self.dt, self.horizon, self.method = dt, horizon, method
        self.key = PRNGKey(0)

    def cost_fn(self, actions, initial_state, goal_local, pcd):
        traj = compute_rollout(actions, initial_state, self.dt)
        dist_cost = jnp.linalg.norm(traj[-1, :2] - goal_local[:2])
        return dist_cost

    def solve(self, initial_state, goal_local, pcd):
        actions = jnp.zeros((self.horizon, 2))
        if self.method == 'GD':
            lr = 0.01
            for _ in range(10):
                actions -= lr * grad(self.cost_fn)(actions, initial_state, goal_local, pcd)
        elif self.method == 'Nesterov':
            v, mu, lr = jnp.zeros_like(actions), 0.9, 0.01
            for _ in range(10):
                g = grad(self.cost_fn)(actions + mu * v, initial_state, goal_local, pcd)
                v = mu * v - lr * g
                actions += v
        elif self.method == 'CEM':
            mean, std, n_samples, n_elite = actions, jnp.ones_like(actions)*0.5, 50, 5
            for _ in range(3):
                self.key, subkey = split(self.key)
                samples = mean + std * normal(subkey, (n_samples, self.horizon, 2))
                costs = vmap(lambda a: self.cost_fn(a, initial_state, goal_local, pcd))(samples)
                elite = samples[jnp.argsort(costs)[:n_elite]]
                mean, std = jnp.mean(elite, axis=0), jnp.std(elite, axis=0) + 1e-4
            actions = mean
        return actions[0], compute_rollout(actions, initial_state, self.dt)